In [ ]:
# 1. suit isolate
import SUITPy as suit
import os
import nibabel as nib
import numpy as np
import smarts_cerebellum.globals as gl

path = os.path.join(gl.baseDir, 'MASSIVE')

# use 2.5mm resolution to match b1000.nii
t1_anat = os.path.join(path, 'Anatatomical data-reg/T1-2.5mm.nii')
save_folder = os.path.join(path, 'processing')

# image given is 4d
img = nib.load(t1_anat)
data = img.get_fdata()
vol3d = data.mean(axis = 3)
img3d = nib.Nifti1Image(vol3d, img.affine)
nib.save(img3d, os.path.join(path, 'processing/T1-2.5mm_3d.nii'))
fixed = os.path.join(path, 'processing/T1-2.5mm_3d.nii')

suit.isolate(t1_file = fixed, result_folder = save_folder, verbose = 0)

In [2]:
# 1. make transformation file (save inverse)
fixed = os.path.join(path, 'processing/T1-2.5mm_3d.nii')
mask_path = os.path.join(path, 'processing/T1-2.5mm_3d_cerebellum_dseg.nii.gz')

template_space = 'MNI152NLin2009cSymC'

results = suit.normalize(
    source_file = fixed,
    mask_file = str(mask_path),
    space = template_space,
    write_inv_deformation = True,

    result_folder = save_folder
    )

# 2. back transform atlas to native space
nettekoven_img = suit.reslice_image(
    'atl-NettekovenSym32_space-MNISym_dseg.nii',
    deformation = results['inv_deformation'],
    interp = 0,
    voxelsize = (2.5, 2.5, 2.5) # match res of T1
    )

nib.save(nettekoven_img, os.path.join(path, 'processing/native_NettekovenSym32_MNISym'))

Normalizing T1-2.5mm_3d to tpl-MNI152NLin2009cSymC_T1w.nii.gz
Saving the normalized image into T1-2.5mm_3d_space-MNI152NLin2009cSymC.nii.gz
Saving deformation field into T1-2.5mm_3d_to-MNI152NLin2009cSymC_mode-image_xfm.nii.gz
Saving inverse deformation field into T1-2.5mm_3d_from-MNI152NLin2009cSymC_mode-image_xfm.nii.gz


FileNotFoundError: No such file or no access: 'atl-NettekovenSym32_space-MNISym_dseg.nii'

In [ ]:
import numpy as np
import nibabel as nib
from nibabel.processing import resample_from_to
from dipy.io.gradients import read_bvals_bvecs # read b values, vectors (b = degree of diffusion weighing)
from dipy.core.gradients import gradient_table # read gradient table
from dipy.reconst.dti import TensorModel # solve Stejskal-Tanner equation for diffusion tensor, D

img = nib.load(path + '/b1000-reg/b1000.nii') # image showing water diffusion along 1000 gradients
data = img.get_fdata()

bvals, bvecs = read_bvals_bvecs(os.path.join(path, 'b1000-reg/b1000.bval'), os.path.join(path, 'b1000-reg/b1000.bvec')) # b value, b vector


# roi
roi = nib.load(os.path.join(path, 'processing/native_NettekovenSym32_MNISym.nii'))
roi_bin = nib.Nifti1Image((roi.get_fdata() > 0).astype(np.float32), roi.affine)
roi_dwi = resample_from_to(roi_bin, (img.shape[:3], img.affine), order=1)
roi_mask = roi_dwi.get_fdata() > 0.5



# fit in only the roi mask
keep = (bvals < 50) | (np.abs(bvals - 1000) < 50)
gtab = gradient_table(bvals[keep], bvecs = bvecs[keep], b0_threshold = 50) # pair b-values with their vectors

fit = TensorModel(gtab, fit_method = 'WLS').fit(data[..., keep], mask = roi_mask) # solve Stejskal-Tanner equation for diffusion tensor D using weighted least squres

fa, v1 = fit.fa, fit.evecs[..., :, 0] # fa, first eigenvector

In [ ]:
from dipy.reconst.dti import color_fa

# colour map for ROI
rgb = color_fa(fa, fit.evecs)
rgb_roi = rgb * roi_mask[..., None] # only inside ROI
nib.save(nib.Nifti1Image(rgb_roi.astype(np.float32), img.affine), path + "/analysis/roi_v1_rgb.nii.gz")